In [ ]:
!tar -xvf ff5da6d6ecae486bb294aeaf5ee8f8a1.tar.gz

Streaming output truncated to the last 5000 lines.
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00019.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00000.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00017.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00010.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00014.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00013.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00005.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00019.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00009.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00007.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00001.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00021.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00000.

In [ ]:
# prompt: help me connect to a google drive so i can upload data once and not everytime

from google.colab import drive
import os, tarfile
drive.mount('/content/drive')

archives = [
    "/content/drive/MyDrive/your_data_folder/0066bbb0f36d497ba1fb2a3bb086fa56.tar.gz",
    "/content/drive/MyDrive/your_data_folder/00674188349c46319d6a3467a23aa183.tar.gz",
    "/content/drive/MyDrive/your_data_folder/008337b68d3d4b16a9fdc3b580b99cc7.tar.gz",
    "/content/drive/MyDrive/your_data_folder/00a34e6ace97444d8594c14d885e3a44.tar.gz",
]

# Step 3: Extract each archive to /content
for archive_path in archives:
    filename = os.path.basename(archive_path)
    print(f"📦 Extracting {filename}...")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path="/content")

print("✅ All archives extracted to /content")

Mounted at /content/drive


In [ ]:
import tarfile

tar_path = "/content/drive/MyDrive/your_data_folder/ff5da6d6ecae486bb294aeaf5ee8f8a1.tar.gz"
extract_path = "/content"

print(f"📦 Extracting {tar_path}...")
with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(path=extract_path)

print("✅ Extraction complete.")



📦 Extracting /content/drive/MyDrive/your_data_folder/ff5da6d6ecae486bb294aeaf5ee8f8a1.tar.gz...
✅ Extraction complete.


In [ ]:
from pathlib import Path
import random

# Path where all extracted folders are located
base_path = Path("/content")

# All 5 folder names (already extracted from tar.gz files)
all_folders = [
    "0066bbb0f36d497ba1fb2a3bb086fa56",
    "00674188349c46319d6a3467a23aa183",
    "008337b68d3d4b16a9fdc3b580b99cc7",
    "00a34e6ace97444d8594c14d885e3a44",
    "ff5da6d6ecae486bb294aeaf5ee8f8a1",
]

# Randomly select one as test, others as train
random.seed(42)  # for reproducibility
test_folder = random.choice(all_folders)
train_folders = [f for f in all_folders if f != test_folder]

(train_folders, test_folder)


(['00674188349c46319d6a3467a23aa183',
  '008337b68d3d4b16a9fdc3b580b99cc7',
  '00a34e6ace97444d8594c14d885e3a44',
  'ff5da6d6ecae486bb294aeaf5ee8f8a1'],
 '0066bbb0f36d497ba1fb2a3bb086fa56')

In [ ]:
from pathlib import Path
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import torch

class VideoAmodalDataset(Dataset):
    def __init__(self, root_dirs, camera_name="camera_0001", size=(256, 256), frames=5):
        self.size = size
        self.frames = frames  # Number of frames (must be odd)
        self.radius = frames // 2
        self.samples = []
        self.resize = transforms.Resize(size)

        for root in root_dirs:
            camera_path = Path(root) / camera_name
            obj_paths = sorted(camera_path.glob("obj_*"))
            for obj_path in obj_paths:
                all_rgbs = sorted(obj_path.glob("rgba_*.png"))
                total = len(all_rgbs)
                for idx in range(self.radius, total - self.radius):
                    self.samples.append({
                        "obj_path": obj_path,
                        "center_idx": idx,
                    })

    def __len__(self):
        return len(self.samples)

    def load_tensor(self, path, mode="RGB"):
        img = Image.open(path).convert(mode)
        return transforms.ToTensor()(self.resize(img))

    def __getitem__(self, idx):
        sample = self.samples[idx]
        obj_path = sample["obj_path"]
        t = sample["center_idx"]

        frames = []
        for offset in range(-self.radius, self.radius + 1):
            frame_id = f"{t + offset:05d}"
            rgb = self.load_tensor(obj_path / f"rgba_{frame_id}.png")
            seg = np.array(Image.open(obj_path / f"segmentation_{frame_id}.png"))

            # Mask generation
            obj_mask = seg.astype(bool)
            scene_overlap = seg[obj_mask]
            labels, counts = np.unique(scene_overlap, return_counts=True)
            object_label = labels[np.argmax(counts)]
            amodal_mask = (seg == object_label).astype(np.float32)
            amodal_mask = torch.from_numpy(amodal_mask).unsqueeze(0)

            masked_rgb = rgb * amodal_mask
            seg_rgb = amodal_mask.repeat(3, 1, 1)
            full_input = torch.cat([amodal_mask, masked_rgb, seg_rgb], dim=0)  # 7 channels

            frames.append(full_input)

        input_tensor = torch.stack(frames, dim=0)  # (F, 7, H, W)

        # Target is center frame's amodal RGB (uncropped)
        target_frame = f"{t:05d}"
        target_rgb = self.load_tensor(obj_path / f"rgba_{target_frame}.png")

        return input_tensor, target_rgb


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)

class VideoUNet(nn.Module):
    def __init__(self, in_channels=35, out_channels=3):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)

        self.upconv4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.final_conv = nn.Conv2d(64, out_channels, 1)
        nn.init.zeros_(self.final_conv.bias)

    def forward(self, x):
        # x: (B, F=5, C=7, H, W)
        B, F, C, H, W = x.shape
        x = x.view(B, F * C, H, W)  # -> (B, 35, H, W)

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.upconv4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        d3 = self.upconv3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.upconv2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.upconv1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.final_conv(d1)
